In [78]:
%load_ext autoreload
%autoreload 2

import os
import joblib
import gc
import time
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tabpfn import TabPFNRegressor
import sys

os.environ["TABPFN_TOKEN"] = "tabpfn_sk_s-vJ41-nt_-nHcf2dZrJ5DTBvU4qG7VGuPdwD4E807g"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    sys.path.insert(0, os.path.abspath(os.getcwd()))

from utils.model_utils import TabPFNQuantileWrapper, SEED, TARGET_COLS
    
device = 'cpu'

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [79]:
def load_and_split_data(train_path, test_path, target_cols):
    """Carica i dati encodati, rimuovendo SEMPRE tutti i possibili target dalle feature."""
    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)
    
    test_ids = df_test['operator_id']
    
    all_possible_targets = [
        'target_assignments', 'target_assN', 'target_assO', 
        'target_assA', 'target_assCP', 'target_assCN', 'target_assMAC'
    ]

    cols_to_drop = all_possible_targets + ['operator_id', 'planning_date']
    
    X_train = df_train.drop(columns=[c for c in cols_to_drop if c in df_train.columns])
    y_train = df_train[target_cols]
    
    X_test = df_test.drop(columns=[c for c in cols_to_drop if c in df_test.columns])
    y_test = df_test[target_cols]
    
    print(f"Dataset caricati: Train ({X_train.shape[0]} righe, {X_train.shape[1]} feature), Test ({X_test.shape[0]} righe)")
    return X_train, y_train, X_test, y_test, test_ids

In [80]:
'''def train_xgboost_quantiles(X_train, y_train, X_test, learning_rate=0.05, max_depth=5, n_estimators=100):
    q10_model = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.1, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, random_state=42)
    q50_model = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.5, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, random_state=42)
    q90_model = xgb.XGBRegressor(objective='reg:quantileerror', quantile_alpha=0.9, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, random_state=42)
    
    q10_model.fit(X_train, y_train)
    q50_model.fit(X_train, y_train)
    q90_model.fit(X_train, y_train)
    
    return (q10_model, q50_model, q90_model), q10_model.predict(X_test), q50_model.predict(X_test), q90_model.predict(X_test)


def train_lightgbm_quantiles(X_train, y_train, X_test, learning_rate=0.05, num_leaves=31, n_estimators=100):
    q10_model = LGBMRegressor(objective='quantile', alpha=0.1, learning_rate=learning_rate, num_leaves=num_leaves, n_estimators=n_estimators, random_state=42, verbose=-1)
    q50_model = LGBMRegressor(objective='quantile', alpha=0.5, learning_rate=learning_rate, num_leaves=num_leaves, n_estimators=n_estimators, random_state=42, verbose=-1)
    q90_model = LGBMRegressor(objective='quantile', alpha=0.9, learning_rate=learning_rate, num_leaves=num_leaves, n_estimators=n_estimators, random_state=42, verbose=-1)
    
    q10_model.fit(X_train, y_train)
    q50_model.fit(X_train, y_train)
    q90_model.fit(X_train, y_train)
    
    return (q10_model, q50_model, q90_model), q10_model.predict(X_test), q50_model.predict(X_test), q90_model.predict(X_test)


def train_sklearn_gb_quantiles(X_train, y_train, X_test, learning_rate=0.05, max_depth=5, n_estimators=100):
    q10_model = GradientBoostingRegressor(loss='quantile', alpha=0.1, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, random_state=42)
    q50_model = GradientBoostingRegressor(loss='quantile', alpha=0.5, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, random_state=42)
    q90_model = GradientBoostingRegressor(loss='quantile', alpha=0.9, learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators, random_state=42)
    
    q10_model.fit(X_train, y_train)
    q50_model.fit(X_train, y_train)
    q90_model.fit(X_train, y_train)
    
    return (q10_model, q50_model, q90_model), q10_model.predict(X_test), q50_model.predict(X_test), q90_model.predict(X_test)
'''

def train_tabpfn_quantiles(X_train, y_train, X_test, n_ensemble=4, device='cpu'):    
    base_model = TabPFNRegressor(n_estimators=n_ensemble, device=device, ignore_pretraining_limits=True)
    base_model.fit(X_train, y_train)
    
    q10_wrapper = TabPFNQuantileWrapper(base_model, 0.1)
    q50_wrapper = TabPFNQuantileWrapper(base_model, 0.5)
    q90_wrapper = TabPFNQuantileWrapper(base_model, 0.9)
    
    q10_pred = q10_wrapper.predict(X_test)
    q50_pred = q50_wrapper.predict(X_test)
    q90_pred = q90_wrapper.predict(X_test)
    
    return (q10_wrapper, q50_wrapper, q90_wrapper), q10_pred, q50_pred, q90_pred

### Ottimizzazione Hyperparametri con Optuna

In [81]:
'''def objective_xgb(trial):
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 9)
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    
    _, _, q50_pred, _ = train_xgboost_quantiles(
        X_train, y_train, X_test, 
        learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators
    )
    return mean_absolute_error(y_test, q50_pred)


def objective_lgb(trial):
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    num_leaves = trial.suggest_int('num_leaves', 15, 127)
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    
    _, _, q50_pred, _ = train_lightgbm_quantiles(
        X_train, y_train, X_test, 
        learning_rate=learning_rate, num_leaves=num_leaves, n_estimators=n_estimators
    )
    return mean_absolute_error(y_test, q50_pred)


def objective_sklearn(trial):
    learning_rate = trial.suggest_float('learning_rate', 0.01, 0.2, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 9)
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    
    _, _, q50_pred, _ = train_sklearn_gb_quantiles(
        X_train, y_train, X_test, 
        learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators
    )
    return mean_absolute_error(y_test, q50_pred)

sampler = TPESampler(seed=SEED)

# 1. XGBoost
study_xgb = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)
print(f"Migliori parametri XGBoost: {study_xgb.best_params} | MAE: {study_xgb.best_value:.4f}")

# 2. LightGBM
study_lgb = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS)
print(f"Migliori parametri LightGBM: {study_lgb.best_params} | MAE: {study_lgb.best_value:.4f}")

# 3. Sklearn GB
study_sk = optuna.create_study(direction='minimize', sampler=TPESampler(seed=SEED))
study_sk.optimize(objective_sklearn, n_trials=N_TRIALS)
print(f"Migliori parametri Sklearn GB: {study_sk.best_params} | MAE: {study_sk.best_value:.4f}")'''

'def objective_xgb(trial):\n    learning_rate = trial.suggest_float(\'learning_rate\', 0.01, 0.2, log=True)\n    max_depth = trial.suggest_int(\'max_depth\', 3, 9)\n    n_estimators = trial.suggest_int(\'n_estimators\', 50, 300)\n\n    _, _, q50_pred, _ = train_xgboost_quantiles(\n        X_train, y_train, X_test, \n        learning_rate=learning_rate, max_depth=max_depth, n_estimators=n_estimators\n    )\n    return mean_absolute_error(y_test, q50_pred)\n\n\ndef objective_lgb(trial):\n    learning_rate = trial.suggest_float(\'learning_rate\', 0.01, 0.2, log=True)\n    num_leaves = trial.suggest_int(\'num_leaves\', 15, 127)\n    n_estimators = trial.suggest_int(\'n_estimators\', 50, 300)\n\n    _, _, q50_pred, _ = train_lightgbm_quantiles(\n        X_train, y_train, X_test, \n        learning_rate=learning_rate, num_leaves=num_leaves, n_estimators=n_estimators\n    )\n    return mean_absolute_error(y_test, q50_pred)\n\n\ndef objective_sklearn(trial):\n    learning_rate = trial.suggest_

In [82]:
def evaluate_model(model_name, y_test, q50, q10, q90):
    mae = mean_absolute_error(y_test, q50)
    rmse = np.sqrt(mean_squared_error(y_test, q50))
    r2 = r2_score(y_test, q50)
    coverage = np.mean((y_test >= q10) & (y_test <= q90))

    print(f"{model_name:<12} | MAE: {mae:.3f} | RMSE: {rmse:.3f} | R^2: {r2:.3f} | Copertura: {coverage*100:.1f}%")
    return mae

In [83]:
'''models_optimized = {
    "XGBoost": lambda X_tr, y_tr, X_te: train_xgboost_quantiles(
        X_tr, y_tr, X_te, **study_xgb.best_params
    ),
    "LightGBM": lambda X_tr, y_tr, X_te: train_lightgbm_quantiles(
        X_tr, y_tr, X_te, **study_lgb.best_params
    ),
    "Sklearn GB": lambda X_tr, y_tr, X_te: train_sklearn_gb_quantiles(
        X_tr, y_tr, X_te, **study_sk.best_params
    )
}'''

X_train, y_train, X_test, y_test, test_ids = load_and_split_data('data/train_dataset.csv', 'data/test_dataset.csv', TARGET_COLS)

X_train_np = X_train.values.astype(np.float32)
X_test_np  = X_test.values.astype(np.float32)

os.makedirs("saved_models", exist_ok=True)

for target in TARGET_COLS:
    print(f"--- Addestramento modello per: {target} ---")
    start_time = time.time()
    
    y_tr_single_np = y_train[target].values.astype(np.float32)
    y_te_single_np = y_test[target].values.astype(np.float32)
    
    trained_wrappers, q10_pred, q50_pred, q90_pred = train_tabpfn_quantiles(
        X_train_np, y_tr_single_np, X_test_np, n_ensemble=2, device=device
    )
    
    evaluate_model(f"TabPFN ({target})", y_te_single_np, q50_pred, q10_pred, q90_pred)
    
    single_target_dict = {
        'q10': trained_wrappers[0],
        'q50': trained_wrappers[1],
        'q90': trained_wrappers[2]
    }
    joblib.dump(single_target_dict, f"saved_models/tabpfn_model_{target}.pkl")
    
    print(f"Completato e salvato in {time.time() - start_time:.2f} sec\n")
    del trained_wrappers, single_target_dict, q10_pred, q50_pred, q90_pred
    gc.collect()

joblib.dump(X_train.columns.tolist(), "saved_models/train_columns.pkl")
print(f"Addestramento concluso! {len(TARGET_COLS)} modelli salvati.")

Dataset caricati: Train (1738 righe, 11 feature), Test (435 righe)
--- Addestramento modello per: target_assignments ---
TabPFN (target_assignments) | MAE: 0.375 | RMSE: 0.623 | R^2: 0.758 | Copertura: 92.4%
Completato e salvato in 7.77 sec

Addestramento concluso! 1 modelli salvati.
